In [ ]:
#!/usr/bin/env python3
"""
RIGOROUS REGULATORY ALIGNMENT AGENT - v3.4 (Publication-Ready)
===============================================================

FIXES FROM v3.3:
----------------
35. ✅ Dynamic CI label in print_results (uses actual alpha)
36. ✅ Calibration leakage fix - calibration regs excluded from training
37. ✅ Apples-to-apples baselines (--baselines-all-runs for CI)
38. ✅ n_test clamping in non-stratified mode (n_test <= n_regs - 1)
39. ✅ Cache sorted(test_indices) to avoid recomputation
40. ✅ Kappa computed on regulation IDs only (excludes ABSTAIN/NO_MATCH)

USAGE:
------
python regulatory_alignment_agent_v3.3.py
python regulatory_alignment_agent_v3.3.py --custom-data path/to/data.csv
python regulatory_alignment_agent_v3.3.py --baselines --runs 10
"""

import os
import sys
import json
import csv
import argparse
import random
import math
import numpy as np
from typing import List, Dict, TypedDict, Tuple, Optional, Set, Any
from dataclasses import dataclass
from enum import Enum
from pathlib import Path
from collections import defaultdict
import warnings
from abc import ABC, abstractmethod

import torch
import torch.nn as nn
from sentence_transformers import (
    SentenceTransformer, InputExample, losses, models, util, CrossEncoder
)
from torch.utils.data import DataLoader
from langgraph.graph import StateGraph, END
from sklearn.metrics import cohen_kappa_score
from sklearn.isotonic import IsotonicRegression

# Optional imports
try:
    from rank_bm25 import BM25Okapi
    BM25_AVAILABLE = True
except ImportError:
    BM25_AVAILABLE = False

try:
    from scipy.stats import t as t_dist
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False


# ==========================================
# CONSTANTS
# ==========================================
ABSTAIN = -1
NO_MATCH = -2
FRAMEWORKS = ['GDPR', 'NIST', 'HIPAA', 'PCI', 'ISO', 'SOX', 'SOC2']


# ==========================================
# STATISTICAL UTILITIES (FIXED v3.4)
# ==========================================
def mean_std_ci(values: List[float], alpha: float = 0.05) -> Tuple[float, float, float]:
    """
    Compute mean, standard deviation, and confidence interval.
    
    FIXED v3.4: Better handling of arbitrary alpha values.
    
    Returns:
        (mean, std, ci_half_width) where CI = [mean - ci, mean + ci]
    """
    vals = np.array(values, dtype=float)
    n = len(vals)
    
    if n == 0:
        return 0.0, 0.0, 0.0
    
    mean = float(np.mean(vals))
    std = float(np.std(vals, ddof=1)) if n > 1 else 0.0
    se = std / math.sqrt(n) if n > 1 else 0.0
    
    # Use t-distribution if scipy available and n > 1
    if SCIPY_AVAILABLE and n > 1:
        tcrit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    elif SCIPY_AVAILABLE:
        # Use normal distribution for any alpha
        from scipy.stats import norm as norm_dist
        tcrit = float(norm_dist.ppf(1 - alpha / 2))
    else:
        # Fallback critical values for common alphas without scipy
        if alpha <= 0.01:
            tcrit = 2.576  # 99% CI
        elif alpha <= 0.05:
            tcrit = 1.96   # 95% CI
        elif alpha <= 0.10:
            tcrit = 1.645  # 90% CI
        else:
            tcrit = 1.96   # Default to 95%
    
    ci = float(tcrit * se)
    return mean, std, ci


# ==========================================
# REPRODUCIBILITY
# ==========================================
def set_all_seeds(seed: int, strict_determinism: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# ==========================================
# CONFIGURATION
# ==========================================
class OperatingMode(Enum):
    CONSERVATIVE = "conservative"
    AGGRESSIVE = "aggressive"
    BALANCED = "balanced"


@dataclass
class RigorousConfig:
    # Models
    LEGALBERT_MODEL: str = 'nlpaueb/legal-bert-base-uncased'
    SECUREBERT_MODEL: str = 'jackaduma/SecRoBERTa'
    CROSS_ENCODER_NAME: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
    GENERIC_MODEL: str = 'sentence-transformers/all-MiniLM-L6-v2'
    
    # Training
    BATCH_SIZE: int = 16
    EPOCHS: int = 8
    WARMUP_STEPS: int = 100
    LEARNING_RATE: float = 5e-6
    
    # Retrieval
    TOP_K_RETRIEVAL: int = 10
    TOP_K_RERANK: int = 5
    
    # Thresholds
    CONSERVATIVE_THRESHOLD: float = 0.75
    BALANCED_THRESHOLD: float = 0.60
    AGGRESSIVE_THRESHOLD: float = 0.45
    
    # Ensemble weights
    LEGALBERT_WEIGHT: float = 0.6
    SECUREBERT_WEIGHT: float = 0.4
    
    # Calibration (FIXED v3.4: separate calibration regulations to prevent leakage)
    USE_CALIBRATION: bool = True
    CALIBRATION_REG_RATIO: float = 0.15  # Fraction of train regs held out for calibration
    
    # Hard negatives
    USE_HARD_NEGATIVES: bool = True
    HARD_NEGATIVES_PER_REG: int = 3
    MINE_HARD_NEGATIVES: bool = True
    
    # Evaluation
    NUM_EVAL_RUNS: int = 5
    HOLDOUT_RATIO: float = 0.2
    USE_ENSEMBLE: bool = True
    EVAL_OPEN_WORLD: bool = False
    CI_ALPHA: float = 0.05  # 95% confidence interval
    
    # Stratified holdout
    USE_STRATIFIED_HOLDOUT: bool = True
    MIN_TEST_PER_FRAMEWORK: int = 1
    
    # Baselines (FIXED v3.4: option for all runs)
    RUN_BM25_BASELINE: bool = True
    RUN_DENSE_ONLY_BASELINE: bool = True
    RUN_SINGLE_DOMAIN_BASELINE: bool = True
    BASELINES_ALL_RUNS: bool = False  # If True, run baselines every run for CI
    
    # Other
    STRICT_DETERMINISM: bool = False
    VERBOSE: bool = True
    
    mode: OperatingMode = OperatingMode.BALANCED
    
    def get_confidence_threshold(self) -> float:
        return {
            OperatingMode.CONSERVATIVE: self.CONSERVATIVE_THRESHOLD,
            OperatingMode.BALANCED: self.BALANCED_THRESHOLD,
            OperatingMode.AGGRESSIVE: self.AGGRESSIVE_THRESHOLD
        }[self.mode]


# ==========================================
# CUSTOM DATA LOADER
# ==========================================
class CustomDataLoader:
    @staticmethod
    def load_from_csv(filepath: str) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        regulations, controls, mappings = {}, {}, []
        control_id_counter = 0
        
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                reg_id = int(row['regulation_id'])
                if reg_id not in regulations:
                    regulations[reg_id] = {'id': reg_id, 'text': row['regulation_text'], 
                                           'framework': row.get('framework', 'UNKNOWN')}
                
                ctrl_text = row['control_text']
                if ctrl_text not in controls:
                    controls[ctrl_text] = {'id': control_id_counter, 'text': ctrl_text,
                                           'family': row.get('control_family', f'family_{control_id_counter}')}
                    control_id_counter += 1
                
                mappings.append({
                    'regulation_id': reg_id, 'control_id': controls[ctrl_text]['id'],
                    'is_match': bool(int(row.get('is_match', 0))),
                    'quality': float(row.get('match_quality', 1.0 if int(row.get('is_match', 0)) else 0.0))
                })
        
        return list(regulations.values()), list(controls.values()), mappings
    
    @staticmethod
    def load_from_json(filepath: str) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data['regulations'], data['controls'], data['mappings']
    
    @staticmethod
    def load(filepath: str) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        path = Path(filepath)
        if not path.exists():
            raise FileNotFoundError(f"Data file not found: {filepath}")
        if path.suffix.lower() == '.csv':
            return CustomDataLoader.load_from_csv(filepath)
        elif path.suffix.lower() == '.json':
            return CustomDataLoader.load_from_json(filepath)
        raise ValueError(f"Unsupported format: {path.suffix}")
    
    @staticmethod
    def validate_data(regulations, controls, mappings) -> Dict[str, Any]:
        reg_ids = {r['id'] for r in regulations}
        ctrl_ids = {c['id'] for c in controls}
        return {
            'n_regulations': len(regulations), 'n_controls': len(controls),
            'n_mappings': len(mappings),
            'n_positive_mappings': sum(1 for m in mappings if m['is_match']),
            'frameworks': list(set(r.get('framework', 'UNKNOWN') for r in regulations)),
            'validation_errors': [f"Unknown reg: {m['regulation_id']}" for m in mappings if m['regulation_id'] not in reg_ids] +
                                [f"Unknown ctrl: {m['control_id']}" for m in mappings if m['control_id'] not in ctrl_ids]
        }
    
    @staticmethod
    def generate_sample_csv(filepath: str = "sample_custom_data.csv"):
        sample_data = [
            {"regulation_id": 0, "regulation_text": "GDPR Art 32: Implement encryption.",
             "control_text": "AES-256 encryption on all databases.", "framework": "GDPR", 
             "is_match": 1, "match_quality": 1.0, "control_family": "gdpr_enc"},
            {"regulation_id": 0, "regulation_text": "GDPR Art 32: Implement encryption.",
             "control_text": "Office supplies ordered.", "framework": "GDPR", 
             "is_match": 0, "match_quality": 0.0, "control_family": "neg"},
        ]
        with open(filepath, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['regulation_id', 'regulation_text', 'control_text', 
                                                    'framework', 'is_match', 'match_quality', 'control_family'])
            writer.writeheader()
            writer.writerows(sample_data)
        print(f"  Sample CSV created: {filepath}")


# ==========================================
# DATA GENERATORS
# ==========================================
class CustomDataGenerator:
    """
    Adapter for custom loaded data.

    v3.4.1 PATCH:
      - Matches ExpandedDataGenerator API:
          generate_training_data_with_holdout -> returns (train_data, cal_reg_indices, actual_train_indices, test_indices)
          get_validation_set -> returns (..., sorted_test)
      - Implements calibration holdout for custom data (no leakage).
      - Applies v3.4 n_test clamp logic in non-stratified mode.
      - Caches sorted_test.
    """

    def __init__(self, regulations: List[Dict], controls: List[Dict], mappings: List[Dict]):
        regs_sorted = sorted(regulations, key=lambda x: x['id'])
        self.all_regulations = [r['text'] for r in regs_sorted]

        # reg position index -> framework
        self.reg_idx_to_framework = {i: r.get('framework', 'UNKNOWN') for i, r in enumerate(regs_sorted)}

        # original regulation_id -> reg position index
        self._orig_id_to_idx = {r['id']: i for i, r in enumerate(regs_sorted)}

        # Build control tuples:
        # (control_text, true_reg_pos_idx or -1, quality_label, match_type, family_id)
        self.all_controls_with_families = []
        mapping_lookup = defaultdict(list)
        for m in mappings:
            mapping_lookup[m['control_id']].append(m)

        for ctrl in sorted(controls, key=lambda x: x['id']):
            ctrl_mappings = mapping_lookup[ctrl['id']]
            best_match, best_quality = None, 0.0
            for m in ctrl_mappings:
                if m['is_match'] and m['quality'] > best_quality:
                    orig_reg_id = m['regulation_id']
                    if orig_reg_id in self._orig_id_to_idx:
                        best_match = self._orig_id_to_idx[orig_reg_id]
                        best_quality = float(m['quality'])

            true_reg = best_match if best_match is not None else -1
            match_type = "perfect" if best_quality >= 0.9 else ("good" if best_quality >= 0.5 else "neg")
            self.all_controls_with_families.append(
                (ctrl['text'], true_reg, best_quality, match_type, ctrl.get('family', f"family_{ctrl['id']}"))
            )

        self._build_lookups()

    def _build_lookups(self):
        self.ctrl_text_to_info = {
            text: (idx, true_reg, label)
            for idx, (text, true_reg, label, _, _) in enumerate(self.all_controls_with_families)
        }
        self.orig_idx_to_reg = {
            idx: true_reg for idx, (_, true_reg, _, _, _) in enumerate(self.all_controls_with_families)
        }
        self.orig_idx_to_label = {
            idx: label for idx, (_, _, label, _, _) in enumerate(self.all_controls_with_families)
        }

    def get_ground_truth_label(self, reg_idx: int, ctrl_orig_idx: int) -> float:
        ctrl_true_reg = self.orig_idx_to_reg.get(ctrl_orig_idx, -1)
        return self.orig_idx_to_label.get(ctrl_orig_idx, 1.0) if ctrl_true_reg == reg_idx else 0.0

    def get_framework_for_regulation(self, reg_idx: int) -> str:
        return self.reg_idx_to_framework.get(reg_idx, 'UNKNOWN')

    def _stratified_split(self, holdout_ratio: float, seed: int, min_per_framework: int) -> Tuple[Set[int], Set[int], List[str]]:
        random.seed(seed)
        warnings_list = []

        framework_regs = defaultdict(list)
        for reg_idx in range(len(self.all_regulations)):
            fw = self.get_framework_for_regulation(reg_idx)
            framework_regs[fw].append(reg_idx)

        train_indices, test_indices = set(), set()

        for fw, regs in framework_regs.items():
            regs_copy = list(regs)
            random.shuffle(regs_copy)

            if len(regs_copy) == 1:
                train_indices.add(regs_copy[0])
                warnings_list.append(f"Framework '{fw}' has only 1 regulation - forced to train set (cannot test)")
                continue

            n_test = max(min_per_framework, int(len(regs_copy) * holdout_ratio))
            n_test = min(n_test, len(regs_copy) - 1)

            test_indices.update(regs_copy[:n_test])
            train_indices.update(regs_copy[n_test:])

        return train_indices, test_indices, warnings_list

    def _mine_hard_negatives(self, train_indices: Set[int], eligible_controls: List[Tuple[int, str, int]],
                             encoder: SentenceTransformer, n_per_reg: int) -> Dict[int, List[Tuple[int, str]]]:
        print("    Mining hard negatives via bi-encoder (custom data)...")
        ctrl_texts = [text for _, text, _ in eligible_controls]
        if not ctrl_texts:
            return {}

        ctrl_embeddings = encoder.encode(
            ctrl_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        hard_neg_map = {}
        for reg_idx in train_indices:
            reg_embedding = encoder.encode(
                self.all_regulations[reg_idx],
                convert_to_tensor=True,
                normalize_embeddings=True,
                show_progress_bar=False
            )
            hits = util.semantic_search(reg_embedding, ctrl_embeddings, top_k=n_per_reg * 3)[0]
            hard_negs = []
            for hit in hits:
                orig_ctrl_idx, ctrl_text, ctrl_true_reg = eligible_controls[hit['corpus_id']]
                if ctrl_true_reg != reg_idx:
                    hard_negs.append((orig_ctrl_idx, ctrl_text))
                if len(hard_negs) >= n_per_reg:
                    break
            hard_neg_map[reg_idx] = hard_negs

        return hard_neg_map

    def generate_training_data_with_holdout(
        self,
        holdout_ratio: float = 0.2,
        seed: int = 42,
        use_hard_negatives: bool = True,
        hard_negatives_per_reg: int = 3,
        mine_hard_negatives: bool = False,
        encoder_for_mining: Optional[SentenceTransformer] = None,
        use_stratified: bool = True,
        min_per_framework: int = 1,
        calibration_reg_ratio: float = 0.15,
    ) -> Tuple[List[InputExample], Set[int], Set[int], Set[int]]:
        """
        v3.4.1:
          - Stratified or non-stratified train/test on regulations
          - Then split TRAIN into CAL + ACTUAL_TRAIN (disjoint) to prevent calibration leakage
          - Returns: (train_data, cal_reg_indices, actual_train_indices, test_indices)
        """
        random.seed(seed)

        if use_stratified:
            all_train_indices, test_indices, warnings_list = self._stratified_split(
                holdout_ratio, seed, min_per_framework
            )
            for w in warnings_list:
                print(f"    ⚠️  {w}")
        else:
            n_regs = len(self.all_regulations)
            if n_regs < 2:
                return [], set(), set(range(n_regs)), set()

            n_test = max(int(n_regs * holdout_ratio), 1)
            # v3.4 clamp: keep at least 1 train reg
            n_test = min(n_test, n_regs - 1)

            indices = list(range(n_regs))
            random.shuffle(indices)
            test_indices = set(indices[:n_test])
            all_train_indices = set(indices[n_test:])

        # Split train regs into calibration regs + actual train regs (disjoint)
        train_list = sorted(all_train_indices)
        if not train_list:
            # extremely defensive fallback
            moved = next(iter(test_indices)) if test_indices else None
            if moved is not None:
                test_indices.remove(moved)
                all_train_indices.add(moved)
                train_list = [moved]

        n_cal = max(1, int(len(train_list) * calibration_reg_ratio)) if len(train_list) > 1 else 0
        random.shuffle(train_list)
        cal_reg_indices = set(train_list[:n_cal]) if n_cal > 0 else set()
        actual_train_indices = set(train_list[n_cal:]) if n_cal > 0 else set(train_list)

        # Hold out families tied to test regs
        holdout_families = {
            family_id
            for _, true_reg, _, _, family_id in self.all_controls_with_families
            if true_reg in test_indices
        }

        # Eligible controls for hard-neg mining
        eligible_for_hard_neg = [
            (orig_idx, text, true_reg)
            for orig_idx, (text, true_reg, _, ctrl_type, family_id) in enumerate(self.all_controls_with_families)
            if family_id not in holdout_families
            and true_reg != -1
            and true_reg in actual_train_indices
            and ctrl_type in ["perfect", "good"]
        ]

        hard_neg_map: Dict[int, List[Tuple[int, str]]] = {}
        if use_hard_negatives and mine_hard_negatives and encoder_for_mining is not None:
            hard_neg_map = self._mine_hard_negatives(
                actual_train_indices, eligible_for_hard_neg, encoder_for_mining, hard_negatives_per_reg
            )

        train_data: List[InputExample] = []
        for _ in range(20):
            for train_idx in actual_train_indices:
                reg_text = self.all_regulations[train_idx]

                for ctrl_text, true_reg, label, _, family_id in self.all_controls_with_families:
                    if family_id in holdout_families:
                        continue
                    if true_reg == train_idx:
                        train_data.append(InputExample(texts=[reg_text, ctrl_text], label=float(label)))
                    elif true_reg == -1:
                        train_data.append(InputExample(texts=[reg_text, ctrl_text], label=0.0))

                if use_hard_negatives and mine_hard_negatives and train_idx in hard_neg_map:
                    for _, ctrl_text in hard_neg_map[train_idx][:hard_negatives_per_reg]:
                        train_data.append(InputExample(texts=[reg_text, ctrl_text], label=0.0))

        random.shuffle(train_data)
        return train_data, cal_reg_indices, actual_train_indices, test_indices

    def get_validation_set(self, test_indices: Set[int], exclude_negatives: bool = True):
        """
        v3.4.1: Return cached sorted_test (parity with v3.4 synthetic path).
        Returns:
          (test_regs, eval_controls, ground_truth, eval_to_reg, ctrl_to_eval_idx, eval_idx_to_orig, sorted_test)
        """
        sorted_test = sorted(test_indices)
        test_regs = [self.all_regulations[i] for i in sorted_test]

        eval_controls, eval_ctrl_orig_indices = [], []
        for orig_idx, (ctrl, true_reg, _, _, _) in enumerate(self.all_controls_with_families):
            if exclude_negatives and true_reg == -1:
                continue
            eval_controls.append(ctrl)
            eval_ctrl_orig_indices.append(orig_idx)

        eval_idx_to_orig = {i: orig for i, orig in enumerate(eval_ctrl_orig_indices)}
        test_idx_to_pos = {idx: pos for pos, idx in enumerate(sorted_test)}

        ground_truth = {}
        for test_idx in test_indices:
            valid_controls = [
                eval_idx
                for eval_idx, orig_idx in enumerate(eval_ctrl_orig_indices)
                if self.all_controls_with_families[orig_idx][1] == test_idx
                and self.all_controls_with_families[orig_idx][3] in ["perfect", "good"]
            ]
            ground_truth[test_idx_to_pos[test_idx]] = valid_controls

        eval_to_reg = {
            eval_idx: self.orig_idx_to_reg.get(orig_idx, -1)
            for eval_idx, orig_idx in enumerate(eval_ctrl_orig_indices)
            if self.orig_idx_to_reg.get(orig_idx, -1) != -1
        }

        return (
            test_regs,
            eval_controls,
            ground_truth,
            eval_to_reg,
            {ctrl: i for i, ctrl in enumerate(eval_controls)},
            eval_idx_to_orig,
            sorted_test,
        )



class ExpandedDataGenerator:
    """Dataset with 50 regulations across 7 frameworks."""
    
    def __init__(self):
        self.all_regulations = [
            # GDPR (0-5)
            "GDPR Art 32: Implement technical measures including encryption of personal data.",
            "GDPR Art 17: Establish procedures for data subject erasure requests.",
            "GDPR Art 33: Notify supervisory authority within 72 hours of breach.",
            "GDPR Art 35: Conduct data protection impact assessments for high-risk processing.",
            "GDPR Art 25: Implement data protection by design and default.",
            "GDPR Art 30: Maintain records of processing activities.",
            # NIST (6-15)
            "NIST AC-2: Establish procedures for account management and review.",
            "NIST AC-6: Implement least privilege access controls.",
            "NIST AU-2: Define auditable events for the system.",
            "NIST AU-6: Review and analyze audit records regularly.",
            "NIST CA-7: Implement continuous monitoring program.",
            "NIST CM-2: Develop and maintain baseline configurations.",
            "NIST CP-9: Conduct regular backups of system data.",
            "NIST IA-2: Implement multi-factor authentication.",
            "NIST IR-4: Implement incident handling procedures.",
            "NIST SC-7: Implement boundary protection controls.",
            # HIPAA (16-23)
            "HIPAA 164.312(a): Implement access controls for ePHI systems.",
            "HIPAA 164.312(b): Implement audit controls to record ePHI activity.",
            "HIPAA 164.312(c): Implement integrity controls for ePHI.",
            "HIPAA 164.312(d): Implement person authentication mechanisms.",
            "HIPAA 164.312(e): Implement transmission security for ePHI.",
            "HIPAA 164.308(a)(1): Conduct risk analysis of ePHI.",
            "HIPAA 164.308(a)(5): Implement security awareness training.",
            "HIPAA 164.310(d): Implement device and media controls.",
            # PCI-DSS (24-33)
            "PCI DSS 1.1: Install and maintain firewall configuration.",
            "PCI DSS 2.1: Change vendor-supplied default passwords.",
            "PCI DSS 3.4: Render PAN unreadable using cryptography.",
            "PCI DSS 4.1: Use strong cryptography for cardholder data transmission.",
            "PCI DSS 5.1: Deploy anti-virus software on systems.",
            "PCI DSS 6.1: Establish process for identifying vulnerabilities.",
            "PCI DSS 7.1: Limit access to cardholder data by business need.",
            "PCI DSS 8.1: Define unique user identification.",
            "PCI DSS 9.1: Restrict physical access to cardholder data.",
            "PCI DSS 10.1: Implement audit trails for system components.",
            # ISO 27001 (34-41)
            "ISO 27001 A.5.1: Establish information security policies.",
            "ISO 27001 A.6.1: Define information security roles.",
            "ISO 27001 A.8.1: Implement asset management procedures.",
            "ISO 27001 A.9.2: Implement user access provisioning.",
            "ISO 27001 A.9.4: Implement access control to systems.",
            "ISO 27001 A.12.1: Document operating procedures.",
            "ISO 27001 A.12.4: Implement logging and monitoring.",
            "ISO 27001 A.13.1: Implement network security controls.",
            # SOX (42-45)
            "SOX 302: Implement controls for financial reporting accuracy.",
            "SOX 404: Document and test internal controls.",
            "SOX 802: Implement record retention policies.",
            "SOX 906: Certify accuracy of financial statements.",
            # SOC 2 (46-49)
            "SOC 2 CC6.1: Implement logical access security software.",
            "SOC 2 CC6.2: Register and authorize users prior to access.",
            "SOC 2 CC6.3: Implement access removal procedures.",
            "SOC 2 CC7.1: Detect and respond to security incidents.",
        ]
        
        # Position-based framework mapping
        self.reg_idx_to_framework = {}
        for i in range(6): self.reg_idx_to_framework[i] = 'GDPR'
        for i in range(6, 16): self.reg_idx_to_framework[i] = 'NIST'
        for i in range(16, 24): self.reg_idx_to_framework[i] = 'HIPAA'
        for i in range(24, 34): self.reg_idx_to_framework[i] = 'PCI'
        for i in range(34, 42): self.reg_idx_to_framework[i] = 'ISO'
        for i in range(42, 46): self.reg_idx_to_framework[i] = 'SOX'
        for i in range(46, 50): self.reg_idx_to_framework[i] = 'SOC2'
        
        self.all_controls_with_families = self._generate_controls()
        self._build_lookups()
    
    def _generate_controls(self):
        controls = []
        # GDPR controls
        controls.extend([
            ("All databases encrypt PII using AES-256 encryption.", 0, 1.0, "perfect", "gdpr_enc_main"),
            ("Production databases use AES-256 for personal data at rest.", 0, 1.0, "perfect", "gdpr_enc_main"),
            ("AWS S3 buckets storing PII have SSE-S3 encryption enabled.", 0, 0.9, "good", "gdpr_enc_cloud"),
            ("Customer data deletion workflow in ServiceNow.", 1, 1.0, "perfect", "gdpr_erasure"),
            ("Automated data subject request portal with 30-day SLA.", 1, 1.0, "perfect", "gdpr_erasure"),
            ("Breach notification procedure documented in runbook.", 2, 1.0, "perfect", "gdpr_breach"),
            ("Security team escalation within 24 hours of incident.", 2, 0.9, "good", "gdpr_breach"),
            ("DPIA template and review process for new projects.", 3, 1.0, "perfect", "gdpr_dpia"),
            ("Privacy by design checklist for development teams.", 4, 1.0, "perfect", "gdpr_pbd"),
            ("Data processing register maintained in OneTrust.", 5, 1.0, "perfect", "gdpr_ropa"),
            ("Encryption at rest using AES-256 for all personal data storage systems.", 0, 1.0, "perfect", "gdpr_crypto_general"),
            ("TLS 1.3 encryption required for all personal data transmission.", 0, 0.9, "good", "gdpr_crypto_general"),
            ("Data subject access request (DSAR) handling procedure with 30-day response.", 1, 1.0, "perfect", "gdpr_dsr_general"),
            ("Right to erasure implementation across all data processing systems.", 1, 1.0, "perfect", "gdpr_dsr_general"),
            ("72-hour breach notification procedure to supervisory authority.", 2, 1.0, "perfect", "gdpr_incident_general"),
            ("Data breach impact assessment and notification workflow.", 2, 0.9, "good", "gdpr_incident_general"),
            ("Privacy impact assessment required for new processing activities.", 3, 1.0, "perfect", "gdpr_privacy_eng"),
            ("Data minimization review in system design phase.", 4, 1.0, "perfect", "gdpr_privacy_eng"),
            ("Privacy by default configuration for all new deployments.", 4, 0.9, "good", "gdpr_privacy_eng"),
            ("Article 30 records of processing maintained in GRC platform.", 5, 1.0, "perfect", "gdpr_documentation"),
            ("Processing activity inventory updated quarterly.", 5, 0.9, "good", "gdpr_documentation"),
        ])
        # NIST controls
        controls.extend([
            ("Quarterly access reviews conducted in ServiceNow.", 6, 1.0, "perfect", "nist_ac2"),
            ("Access recertification campaign every 90 days.", 6, 1.0, "perfect", "nist_ac2"),
            ("Role-based access with manager approval workflow.", 7, 1.0, "perfect", "nist_ac6"),
            ("Privileged access requires additional approval.", 7, 0.9, "good", "nist_ac6"),
            ("Audit log configuration for authentication events.", 8, 1.0, "perfect", "nist_au2"),
            ("SIEM correlation rules for critical events.", 9, 1.0, "perfect", "nist_au6"),
            ("Weekly security log review by SOC team.", 9, 0.9, "good", "nist_au6"),
            ("Continuous vulnerability scanning with Qualys.", 10, 1.0, "perfect", "nist_ca7"),
            ("Golden image maintained for server deployments.", 11, 1.0, "perfect", "nist_cm2"),
            ("Daily encrypted backups to offsite location.", 12, 1.0, "perfect", "nist_cp9"),
            ("MFA required for all remote access.", 13, 1.0, "perfect", "nist_ia2"),
            ("Okta MFA for VPN and cloud applications.", 13, 1.0, "perfect", "nist_ia2"),
            ("Incident response playbook with escalation matrix.", 14, 1.0, "perfect", "nist_ir4"),
            ("Network segmentation with firewall rules.", 15, 1.0, "perfect", "nist_sc7"),
        ])
        # HIPAA controls
        controls.extend([
            ("EHR access restricted by role and department.", 16, 1.0, "perfect", "hipaa_access"),
            ("Break-glass procedure for emergency ePHI access.", 16, 0.9, "good", "hipaa_access"),
            ("All ePHI access logged to SIEM.", 17, 1.0, "perfect", "hipaa_audit"),
            ("EHR audit logs retained for 6 years.", 17, 1.0, "perfect", "hipaa_audit"),
            ("Database integrity checks via checksums.", 18, 1.0, "perfect", "hipaa_integrity"),
            ("Biometric authentication for clinical systems.", 19, 1.0, "perfect", "hipaa_auth"),
            ("TLS 1.3 for all ePHI transmission.", 20, 1.0, "perfect", "hipaa_trans"),
            ("Annual HIPAA risk assessment documented.", 21, 1.0, "perfect", "hipaa_risk"),
            ("Quarterly security awareness training for staff.", 22, 1.0, "perfect", "hipaa_training"),
            ("Media sanitization procedure for decommissioned devices.", 23, 1.0, "perfect", "hipaa_media"),
        ])
        # PCI controls
        controls.extend([
            ("Palo Alto firewall with documented ruleset.", 24, 1.0, "perfect", "pci_firewall"),
            ("Default passwords changed during deployment.", 25, 1.0, "perfect", "pci_defaults"),
            ("PAN tokenized before storage in database.", 26, 1.0, "perfect", "pci_pan"),
            ("Card data hashed with SHA-256 salt.", 26, 1.0, "perfect", "pci_pan"),
            ("TLS 1.2+ required for payment processing.", 27, 1.0, "perfect", "pci_crypto"),
            ("CrowdStrike endpoint protection deployed.", 28, 1.0, "perfect", "pci_av"),
            ("Weekly vulnerability scans with Nessus.", 29, 1.0, "perfect", "pci_vuln"),
            ("Cardholder data access limited to payment team.", 30, 1.0, "perfect", "pci_access"),
            ("Unique user IDs for all system access.", 31, 1.0, "perfect", "pci_users"),
            ("Badge access to data center.", 32, 1.0, "perfect", "pci_physical"),
            ("Comprehensive audit logging enabled.", 33, 1.0, "perfect", "pci_audit"),
        ])
        # ISO controls
        controls.extend([
            ("Information security policy reviewed annually.", 34, 1.0, "perfect", "iso_policy"),
            ("CISO reports to executive committee.", 35, 1.0, "perfect", "iso_roles"),
            ("Asset inventory maintained in CMDB.", 36, 1.0, "perfect", "iso_assets"),
            ("User provisioning via identity management system.", 37, 1.0, "perfect", "iso_prov"),
            ("HR triggers access provisioning workflow.", 37, 0.9, "good", "iso_prov"),
            ("System access requires manager approval.", 38, 1.0, "perfect", "iso_access"),
            ("Runbooks documented for all critical systems.", 39, 1.0, "perfect", "iso_docs"),
            ("Centralized logging with 90-day retention.", 40, 1.0, "perfect", "iso_logging"),
            ("Network IDS/IPS at perimeter.", 41, 1.0, "perfect", "iso_network"),
        ])
        # SOX controls
        controls.extend([
            ("Financial system access reviewed quarterly.", 42, 1.0, "perfect", "sox_302"),
            ("SOX control testing documented annually.", 43, 1.0, "perfect", "sox_404"),
            ("Financial records retained 7 years.", 44, 1.0, "perfect", "sox_802"),
            ("CFO certification of quarterly financials.", 45, 1.0, "perfect", "sox_906"),
            ("Segregation of duties enforced for financial transactions.", 42, 1.0, "perfect", "sox_general"),
            ("Internal control documentation maintained in GRC system.", 43, 1.0, "perfect", "sox_general"),
            ("Management assertion on internal controls effectiveness.", 43, 0.9, "good", "sox_general"),
            ("Document retention policy for financial records.", 44, 1.0, "perfect", "sox_retention"),
            ("Quarterly financial certification by executive management.", 45, 1.0, "perfect", "sox_certification"),
        ])
        # SOC2 controls
        controls.extend([
            ("Logical access security via Okta.", 46, 1.0, "perfect", "soc2_cc61"),
            ("User registration requires manager approval.", 47, 1.0, "perfect", "soc2_cc62"),
            ("Automated deprovisioning on termination.", 48, 1.0, "perfect", "soc2_cc63"),
            ("SIEM alerting for security events.", 49, 1.0, "perfect", "soc2_cc71"),
            ("Access control policy enforced through IAM system.", 46, 1.0, "perfect", "soc2_access_general"),
            ("Principle of least privilege implemented for all users.", 46, 0.9, "good", "soc2_access_general"),
            ("User onboarding workflow with approval chain.", 47, 1.0, "perfect", "soc2_user_mgmt"),
            ("Termination checklist includes access revocation.", 48, 1.0, "perfect", "soc2_user_mgmt"),
            ("Security incident detection and response procedures.", 49, 1.0, "perfect", "soc2_incident"),
            ("24/7 security monitoring and alerting.", 49, 0.9, "good", "soc2_incident"),
        ])
        # Negatives
        controls.extend([
            ("Office supplies ordered monthly.", -1, 0.0, "neg", "neg_general"),
            ("Team lunch scheduled for Friday.", -1, 0.0, "neg", "neg_general"),
            ("Marketing campaign launched.", -1, 0.0, "neg", "neg_general"),
            ("New hire orientation completed.", -1, 0.0, "neg", "neg_hr"),
            ("Quarterly business review meeting.", -1, 0.0, "neg", "neg_business"),
            ("Product roadmap updated.", -1, 0.0, "neg", "neg_product"),
            ("Customer feedback survey sent.", -1, 0.0, "neg", "neg_customer"),
            ("Annual company retreat planned.", -1, 0.0, "neg", "neg_hr"),
        ])
        return controls
    
    def _build_lookups(self):
        self.ctrl_text_to_info = {text: (idx, true_reg, label) for idx, (text, true_reg, label, _, _) in enumerate(self.all_controls_with_families)}
        self.orig_idx_to_reg = {idx: true_reg for idx, (_, true_reg, _, _, _) in enumerate(self.all_controls_with_families)}
        self.orig_idx_to_label = {idx: label for idx, (_, _, label, _, _) in enumerate(self.all_controls_with_families)}
    
    def get_ground_truth_label(self, reg_idx: int, ctrl_orig_idx: int) -> float:
        ctrl_true_reg = self.orig_idx_to_reg.get(ctrl_orig_idx, -1)
        return self.orig_idx_to_label.get(ctrl_orig_idx, 1.0) if ctrl_true_reg == reg_idx else 0.0
    
    def get_framework_for_regulation(self, reg_idx: int) -> str:
        return self.reg_idx_to_framework.get(reg_idx, 'UNKNOWN')
    
    def generate_training_data_with_holdout(self, holdout_ratio=0.2, seed=42, use_hard_negatives=True,
                                           hard_negatives_per_reg=3, mine_hard_negatives=False,
                                           encoder_for_mining=None, use_stratified=True, min_per_framework=1,
                                           calibration_reg_ratio=0.15):
        """
        FIXED v3.4: Proper calibration holdout to prevent leakage.
        Returns (train_data, cal_reg_indices, actual_train_indices, test_indices)
        """
        random.seed(seed)
        
        if use_stratified:
            all_train_indices, test_indices, warnings_list = self._stratified_split(holdout_ratio, seed, min_per_framework)
            for w in warnings_list:
                print(f"    ⚠️  {w}")
        else:
            n_regs = len(self.all_regulations)
            n_test = max(int(n_regs * holdout_ratio), 10)
            # FIX v3.4: Clamp to ensure non-empty train
            n_test = min(n_test, n_regs - 1)
            indices = list(range(n_regs))
            random.shuffle(indices)
            test_indices, all_train_indices = set(indices[:n_test]), set(indices[n_test:])
        
        # FIX v3.4: Split train into actual train and calibration (disjoint)
        train_list = sorted(all_train_indices)
        n_cal = max(1, int(len(train_list) * calibration_reg_ratio))
        random.shuffle(train_list)
        cal_reg_indices = set(train_list[:n_cal])
        actual_train_indices = set(train_list[n_cal:])
        
        holdout_families = {family_id for _, true_reg, _, _, family_id in self.all_controls_with_families 
                          if true_reg in test_indices}
        
        eligible_for_hard_neg = [(idx, text, true_reg) for idx, (text, true_reg, _, ctrl_type, family_id) 
                                  in enumerate(self.all_controls_with_families) 
                                  if family_id not in holdout_families and true_reg != -1 
                                  and true_reg in actual_train_indices and ctrl_type in ["perfect", "good"]]
        
        hard_neg_map = {}
        if use_hard_negatives and mine_hard_negatives and encoder_for_mining is not None:
            hard_neg_map = self._mine_hard_negatives(actual_train_indices, eligible_for_hard_neg, 
                                                     encoder_for_mining, hard_negatives_per_reg)
        
        train_data = []
        for _ in range(20):
            for train_idx in actual_train_indices:  # Exclude calibration regs from training
                reg_text = self.all_regulations[train_idx]
                for ctrl_text, true_reg, label, _, family_id in self.all_controls_with_families:
                    if family_id in holdout_families:
                        continue
                    if true_reg == train_idx:
                        train_data.append(InputExample(texts=[reg_text, ctrl_text], label=label))
                    elif true_reg == -1:
                        train_data.append(InputExample(texts=[reg_text, ctrl_text], label=0.0))
                
                if use_hard_negatives and mine_hard_negatives and train_idx in hard_neg_map:
                    for _, ctrl_text in hard_neg_map[train_idx][:hard_negatives_per_reg]:
                        train_data.append(InputExample(texts=[reg_text, ctrl_text], label=0.0))
        
        random.shuffle(train_data)
        return (train_data, cal_reg_indices, actual_train_indices, test_indices)
    
    def _stratified_split(self, holdout_ratio: float, seed: int, min_per_framework: int) -> Tuple[Set[int], Set[int], List[str]]:
        """Stratified split with edge case handling for single-regulation frameworks."""
        random.seed(seed)
        warnings_list = []
        
        framework_regs = defaultdict(list)
        for reg_idx in range(len(self.all_regulations)):
            fw = self.get_framework_for_regulation(reg_idx)
            framework_regs[fw].append(reg_idx)
        
        train_indices, test_indices = set(), set()
        
        for fw, regs in framework_regs.items():
            regs_copy = list(regs)
            random.shuffle(regs_copy)
            
            if len(regs_copy) == 1:
                train_indices.add(regs_copy[0])
                warnings_list.append(f"Framework '{fw}' has only 1 regulation - forced to train set")
                continue
            
            n_test = max(min_per_framework, int(len(regs_copy) * holdout_ratio))
            n_test = min(n_test, len(regs_copy) - 1)
            
            test_indices.update(regs_copy[:n_test])
            train_indices.update(regs_copy[n_test:])
        
        return train_indices, test_indices, warnings_list
    
    def _mine_hard_negatives(self, train_indices, eligible_controls, encoder, n_per_reg):
        print("    Mining hard negatives via bi-encoder...")
        ctrl_texts = [text for _, text, _ in eligible_controls]
        ctrl_embeddings = encoder.encode(ctrl_texts, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
        
        hard_neg_map = {}
        for reg_idx in train_indices:
            reg_embedding = encoder.encode(self.all_regulations[reg_idx], convert_to_tensor=True, normalize_embeddings=True)
            hits = util.semantic_search(reg_embedding, ctrl_embeddings, top_k=n_per_reg * 3)[0]
            hard_negs = [(eligible_controls[hit['corpus_id']][0], eligible_controls[hit['corpus_id']][1]) 
                        for hit in hits if eligible_controls[hit['corpus_id']][2] != reg_idx][:n_per_reg]
            hard_neg_map[reg_idx] = hard_negs
        return hard_neg_map
    
    def get_validation_set(self, test_indices, exclude_negatives: bool = True):
        # FIX v3.4: Cache sorted_test and return it
        sorted_test = sorted(test_indices)
        test_regs = [self.all_regulations[i] for i in sorted_test]
        eval_controls, eval_ctrl_orig_indices = [], []
        
        for orig_idx, (ctrl, true_reg, _, _, _) in enumerate(self.all_controls_with_families):
            if exclude_negatives and true_reg == -1:
                continue
            eval_controls.append(ctrl)
            eval_ctrl_orig_indices.append(orig_idx)
        
        eval_idx_to_orig = {i: orig for i, orig in enumerate(eval_ctrl_orig_indices)}
        
        # Precompute mapping for O(1) lookups
        test_idx_to_pos = {idx: pos for pos, idx in enumerate(sorted_test)}
        
        ground_truth = {}
        for test_idx in test_indices:
            valid_controls = [eval_idx for eval_idx, orig_idx in enumerate(eval_ctrl_orig_indices) 
                            if self.all_controls_with_families[orig_idx][1] == test_idx 
                            and self.all_controls_with_families[orig_idx][3] in ["perfect", "good"]]
            ground_truth[test_idx_to_pos[test_idx]] = valid_controls
        
        eval_to_reg = {eval_idx: self.orig_idx_to_reg[orig_idx] for eval_idx, orig_idx in enumerate(eval_ctrl_orig_indices) 
                      if self.orig_idx_to_reg.get(orig_idx, -1) != -1}
        
        return test_regs, eval_controls, ground_truth, eval_to_reg, \
               {ctrl: i for i, ctrl in enumerate(eval_controls)}, eval_idx_to_orig, sorted_test


# ==========================================
# BASELINE RETRIEVERS
# ==========================================
class BaseRetriever(ABC):
    @abstractmethod
    def ingest_controls(self, controls: List[str]): pass
    @abstractmethod
    def retrieve(self, query: str, top_k: int = 10) -> List[Dict]: pass
    @property
    @abstractmethod
    def name(self) -> str: pass


class BM25Retriever(BaseRetriever):
    def __init__(self):
        if not BM25_AVAILABLE:
            raise ImportError("rank_bm25 not installed")
        self.bm25 = None
        self.control_index = []
    
    @property
    def name(self) -> str:
        return "BM25"
    
    def ingest_controls(self, controls: List[str]):
        self.control_index = controls
        self.bm25 = BM25Okapi([doc.lower().split() for doc in controls])
    
    def retrieve(self, query: str, top_k: int = 10) -> List[Dict]:
        scores = self.bm25.get_scores(query.lower().split())
        top_indices = np.argsort(scores)[::-1][:top_k]
        max_score = max(scores) + 1e-10
        return [{"control_id": int(idx), "control_text": self.control_index[idx], 
                "score": float(scores[idx]), "final_score": float(scores[idx]) / max_score} for idx in top_indices]


class DenseOnlyRetriever(BaseRetriever):
    def __init__(self, model_name: str = 'sentence-transformers/all-MiniLM-L6-v2'):
        self.encoder = SentenceTransformer(model_name)
        self.control_index = []
        self.embeddings = None
        self._name = f"Dense-Only ({model_name.split('/')[-1]})"
    
    @property
    def name(self) -> str:
        return self._name
    
    def ingest_controls(self, controls: List[str]):
        self.control_index = controls
        self.embeddings = self.encoder.encode(controls, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    
    def retrieve(self, query: str, top_k: int = 10) -> List[Dict]:
        query_emb = self.encoder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
        hits = util.semantic_search(query_emb, self.embeddings, top_k=top_k)[0]
        return [{"control_id": hit['corpus_id'], "control_text": self.control_index[hit['corpus_id']], 
                "score": float(hit['score']), "final_score": (float(hit['score']) + 1) / 2} for hit in hits]


class SingleDomainRetriever(BaseRetriever):
    def __init__(self, model_name: str, domain_name: str = "single-domain"):
        transformer = models.Transformer(model_name, max_seq_length=256)
        pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode='mean')
        self.encoder = SentenceTransformer(modules=[transformer, pooling])
        self.control_index = []
        self.embeddings = None
        self._name = f"Single-Domain ({domain_name})"
    
    @property
    def name(self) -> str:
        return self._name
    
    def ingest_controls(self, controls: List[str]):
        self.control_index = controls
        self.embeddings = self.encoder.encode(controls, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    
    def retrieve(self, query: str, top_k: int = 10) -> List[Dict]:
        query_emb = self.encoder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
        hits = util.semantic_search(query_emb, self.embeddings, top_k=top_k)[0]
        return [{"control_id": hit['corpus_id'], "control_text": self.control_index[hit['corpus_id']], 
                "score": float(hit['score']), "final_score": (float(hit['score']) + 1) / 2} for hit in hits]


# ==========================================
# DUAL MODEL FINE-TUNER
# ==========================================
class DualModelFineTuner:
    def __init__(self, use_ensemble=True, config=None):
        self.config = config or RigorousConfig()
        self.use_ensemble = use_ensemble
        
        legal_transformer = models.Transformer(self.config.LEGALBERT_MODEL, max_seq_length=256)
        legal_pooling = models.Pooling(legal_transformer.get_word_embedding_dimension(), pooling_mode='mean')
        self.legal_model = SentenceTransformer(modules=[legal_transformer, legal_pooling])
        
        if use_ensemble:
            try:
                secure_transformer = models.Transformer(self.config.SECUREBERT_MODEL, max_seq_length=256)
                secure_pooling = models.Pooling(secure_transformer.get_word_embedding_dimension(), pooling_mode='mean')
                self.secure_model = SentenceTransformer(modules=[secure_transformer, secure_pooling])
            except Exception:
                self.use_ensemble = False
                self.secure_model = None
        else:
            self.secure_model = None
    
    def train(self, train_examples):
        train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=self.config.BATCH_SIZE)
        
        legal_loss = losses.CosineSimilarityLoss(self.legal_model)
        self.legal_model.fit(train_objectives=[(train_dataloader, legal_loss)], epochs=self.config.EPOCHS,
                            warmup_steps=self.config.WARMUP_STEPS, optimizer_params={'lr': self.config.LEARNING_RATE},
                            show_progress_bar=False)
        
        if self.use_ensemble and self.secure_model:
            secure_loss = losses.CosineSimilarityLoss(self.secure_model)
            self.secure_model.fit(train_objectives=[(train_dataloader, secure_loss)], epochs=self.config.EPOCHS,
                                 warmup_steps=self.config.WARMUP_STEPS, optimizer_params={'lr': self.config.LEARNING_RATE},
                                 show_progress_bar=False)
        
        return self.legal_model, self.secure_model


# ==========================================
# RAG SYSTEM
# ==========================================
class RigorousRAG:
    def __init__(self, legal_model, secure_model=None, config=None):
        self.config = config or RigorousConfig()
        self.legal_encoder = legal_model
        self.secure_encoder = secure_model
        self.use_ensemble = secure_model is not None
        self.cross_encoder = CrossEncoder(self.config.CROSS_ENCODER_NAME)
        self.calibrator = IsotonicRegression(out_of_bounds='clip') if self.config.USE_CALIBRATION else None
        self._calibrator_fitted = False
        self.control_index = []
        self.legal_embeddings = None
        self.secure_embeddings = None
    
    def ingest_controls(self, controls: List[str]):
        self.control_index = controls
        self.legal_embeddings = self.legal_encoder.encode(controls, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
        if self.use_ensemble and self.secure_encoder:
            self.secure_embeddings = self.secure_encoder.encode(controls, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    
    def _rescale_bi_score(self, bi_score: float) -> float:
        return float(np.clip((bi_score + 1.0) / 2.0, 0.0, 1.0))
    
    def _retrieve_candidates_with_scores(self, regulation_text: str) -> List[Dict]:
        legal_query_emb = self.legal_encoder.encode(regulation_text, convert_to_tensor=True, normalize_embeddings=True)
        legal_hits = util.semantic_search(legal_query_emb, self.legal_embeddings, top_k=self.config.TOP_K_RETRIEVAL)[0]
        
        secure_hits = []
        if self.use_ensemble and self.secure_embeddings is not None:
            secure_query_emb = self.secure_encoder.encode(regulation_text, convert_to_tensor=True, normalize_embeddings=True)
            secure_hits = util.semantic_search(secure_query_emb, self.secure_embeddings, top_k=self.config.TOP_K_RETRIEVAL)[0]
        
        combined_scores = {}
        for hit in legal_hits:
            combined_scores[hit['corpus_id']] = {'control_id': hit['corpus_id'], 'legal_score': float(hit['score']), 'secure_score': 0.0}
        for hit in secure_hits:
            if hit['corpus_id'] in combined_scores:
                combined_scores[hit['corpus_id']]['secure_score'] = float(hit['score'])
            else:
                combined_scores[hit['corpus_id']] = {'control_id': hit['corpus_id'], 'legal_score': 0.0, 'secure_score': float(hit['score'])}
        
        for ctrl_id, scores in combined_scores.items():
            raw_bi = (self.config.LEGALBERT_WEIGHT * scores['legal_score'] + self.config.SECUREBERT_WEIGHT * scores['secure_score']) if self.use_ensemble else scores['legal_score']
            scores['bi_score'] = self._rescale_bi_score(raw_bi)
        
        return sorted(combined_scores.values(), key=lambda x: x['bi_score'], reverse=True)[:self.config.TOP_K_RETRIEVAL]
    
    def calibrate_on_retrieval_distribution(self, cal_reg_indices: List[int], data_gen, eval_idx_to_orig: Dict[int, int]):
        if not self.config.USE_CALIBRATION:
            return
        
        calibration_pairs = []
        for reg_idx in cal_reg_indices:
            reg_text = data_gen.all_regulations[reg_idx]
            for candidate in self._retrieve_candidates_with_scores(reg_text):
                orig_ctrl_idx = eval_idx_to_orig.get(candidate['control_id'])
                if orig_ctrl_idx is not None:
                    calibration_pairs.append((reg_text, self.control_index[candidate['control_id']], 
                                             data_gen.get_ground_truth_label(reg_idx, orig_ctrl_idx)))
        
        if len(calibration_pairs) < 20:
            return
        
        pairs = [[reg, ctrl] for reg, ctrl, _ in calibration_pairs]
        raw_scores = self.cross_encoder.predict(pairs, activation_fct=nn.Identity(), show_progress_bar=False)
        self.calibrator.fit(raw_scores, np.array([label for _, _, label in calibration_pairs]))
        self._calibrator_fitted = True
        print(f"    [Calibrator fitted on {len(calibration_pairs)} pairs]")
    
    def retrieve_and_rank(self, regulation_text: str) -> List[Dict]:
        candidates = self._retrieve_candidates_with_scores(regulation_text)
        if not candidates:
            return []
        
        candidate_pairs = [[regulation_text, self.control_index[c['control_id']]] for c in candidates]
        cross_scores_raw = self.cross_encoder.predict(candidate_pairs, activation_fct=nn.Identity(), show_progress_bar=False)
        cross_scores = self.calibrator.transform(cross_scores_raw) if self.calibrator and self._calibrator_fitted else 1 / (1 + np.exp(-cross_scores_raw))
        
        results = []
        for i, candidate in enumerate(candidates):
            cross_score = float(np.clip(cross_scores[i], 0.0, 1.0))
            bi_score = candidate['bi_score']
            
            if self.config.mode == OperatingMode.CONSERVATIVE:
                final_score = 0.3 * bi_score + 0.7 * cross_score
            elif self.config.mode == OperatingMode.AGGRESSIVE:
                final_score = 0.7 * bi_score + 0.3 * cross_score
            else:
                final_score = 0.5 * bi_score + 0.5 * cross_score
            
            results.append({"control_id": candidate['control_id'], "control_text": self.control_index[candidate['control_id']], 
                           "bi_score": bi_score, "cross_score": cross_score, "final_score": final_score})
        
        return sorted(results, key=lambda x: x['final_score'], reverse=True)[:self.config.TOP_K_RERANK]


# ==========================================
# AGENT
# ==========================================
class RigorousAgentState(TypedDict):
    regulation: str
    regulation_id: int
    ranked_results: List[Dict]
    best_match: Optional[Dict]
    confidence: float
    status: str


class RigorousAgent:
    def __init__(self, rag_system: RigorousRAG, config=None):
        self.rag = rag_system
        self.config = config or RigorousConfig()
        self.workflow = self._build_graph()
    
    def _retrieve_node(self, state: RigorousAgentState):
        results = self.rag.retrieve_and_rank(state['regulation'])
        return {"ranked_results": results, "best_match": results[0] if results else None}
    
    def _reflector_node(self, state: RigorousAgentState):
        best = state['best_match']
        if not best:
            return {"confidence": 0.0, "status": "abstain"}
        conf = best['final_score']
        return {"confidence": conf, "status": "high_confidence" if conf >= self.config.get_confidence_threshold() else "abstain"}
    
    def _build_graph(self):
        workflow = StateGraph(RigorousAgentState)
        workflow.add_node("retrieve", self._retrieve_node)
        workflow.add_node("reflector", self._reflector_node)
        workflow.set_entry_point("retrieve")
        workflow.add_edge("retrieve", "reflector")
        workflow.add_edge("reflector", END)
        return workflow.compile()
    
    def run(self, regulation_text: str, regulation_id: int):
        return self.workflow.invoke({"regulation": regulation_text, "regulation_id": regulation_id, 
                                     "ranked_results": [], "best_match": None, "confidence": 0.0, "status": ""})


# ==========================================
# METRICS (FIXED v3.3)
# ==========================================
class RigorousMetrics:
    @staticmethod
    def calculate_recall_at_k(ground_truth_map: Dict, predictions_map: Dict, k: int = 5) -> List[float]:
        """Returns list of per-query recall values for CI calculation."""
        return [len(set(true_ids) & set(predictions_map[reg_id][:k])) / len(true_ids) 
                for reg_id, true_ids in ground_truth_map.items() 
                if reg_id in predictions_map and len(true_ids) > 0]
    
    @staticmethod
    def calculate_precision_at_k(ground_truth_map: Dict, predictions_map: Dict, k: int = 5) -> List[float]:
        """Returns list of per-query precision values for CI calculation."""
        return [len(set(true_ids) & set(predictions_map[reg_id][:k])) / len(predictions_map[reg_id][:k]) 
                for reg_id, true_ids in ground_truth_map.items() 
                if reg_id in predictions_map and len(predictions_map[reg_id][:k]) > 0]
    
    @staticmethod
    def calculate_f1(recall: float, precision: float) -> float:
        return 2 * (precision * recall) / (precision + recall) if recall + precision > 0 else 0.0
    
    @staticmethod
    def calculate_ndcg_at_k(ground_truth_map: Dict, predictions_map: Dict, k: int = 5) -> List[float]:
        """
        FIXED v3.3: Removed unused scores_map parameter.
        NDCG uses the order of predictions (which is already sorted by score).
        
        Returns list of per-query NDCG values for CI calculation.
        """
        ndcg_scores = []
        
        for reg_id, true_ids in ground_truth_map.items():
            if reg_id not in predictions_map or len(true_ids) == 0:
                continue
            
            pred_ids = predictions_map[reg_id][:k]
            if len(pred_ids) == 0:
                continue
            
            # Binary relevance based on ground truth
            relevances = [1.0 if pid in true_ids else 0.0 for pid in pred_ids]
            
            # DCG using position in the ranked list
            dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances))
            
            # IDCG: ideal would have all relevant items first
            n_relevant = min(len(true_ids), k)
            idcg = sum(1.0 / math.log2(i + 2) for i in range(n_relevant))
            
            if idcg > 0:
                ndcg_scores.append(dcg / idcg)
        
        return ndcg_scores
    
    @staticmethod
    def calculate_kappa_on_regulations(y_true_regs: List[int], y_pred_regs: List[int]) -> float:
        """
        FIXED v3.4: Kappa computed on regulation IDs only, excluding ABSTAIN/NO_MATCH.
        Returns kappa over the subset where predictions were made.
        """
        paired = [(t, p) for t, p in zip(y_true_regs, y_pred_regs) if p not in [ABSTAIN, NO_MATCH]]
        
        if len(paired) < 2:
            return 0.0
        
        y_true_filtered = [p[0] for p in paired]
        y_pred_filtered = [p[1] for p in paired]
        
        try:
            labels = sorted(set(y_true_filtered) | set(y_pred_filtered))
            return cohen_kappa_score(y_true_filtered, y_pred_filtered, labels=labels)
        except:
            return 0.0
    
    @staticmethod
    def calculate_coverage_and_accuracy(y_true: List[int], y_pred: List[int]) -> Tuple[float, float, int, int]:
        """Calculate coverage, selective accuracy, abstention count, and no-match count."""
        y_pred_arr = np.array(y_pred)
        y_true_arr = np.array(y_true)
        
        n_abstain = int(np.sum(y_pred_arr == ABSTAIN))
        n_no_match = int(np.sum(y_pred_arr == NO_MATCH))
        
        made_prediction = (y_pred_arr != ABSTAIN) & (y_pred_arr != NO_MATCH)
        coverage = np.sum(made_prediction) / len(y_pred_arr) if len(y_pred_arr) > 0 else 0.0
        
        if np.sum(made_prediction) > 0:
            selective_acc = np.sum(y_true_arr[made_prediction] == y_pred_arr[made_prediction]) / np.sum(made_prediction)
        else:
            selective_acc = 0.0
        
        return float(coverage), float(selective_acc), n_abstain, n_no_match
    
    @staticmethod
    def calculate_metrics_with_abstention(y_true, y_pred, eval_to_reg: Dict):
        """Legacy method for backward compatibility."""
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        
        # Use new kappa calculation
        kappa = RigorousMetrics.calculate_kappa_on_regulations(list(y_true), list(y_pred))
        
        made_prediction = (y_pred != ABSTAIN) & (y_pred != NO_MATCH)
        coverage = np.sum(made_prediction) / len(y_pred) if len(y_pred) > 0 else 0.0
        selective_acc = np.sum(y_true[made_prediction] == y_pred[made_prediction]) / np.sum(made_prediction) if np.sum(made_prediction) > 0 else 0.0
        misassign_rate = 1.0 - selective_acc
        
        return kappa, coverage, selective_acc, int(np.sum(y_pred == ABSTAIN)), int(np.sum(y_pred == NO_MATCH)), misassign_rate


# ==========================================
# PER-FRAMEWORK EVALUATOR
# ==========================================
class PerFrameworkEvaluator:
    def __init__(self, data_gen, config: RigorousConfig):
        self.data_gen = data_gen
        self.config = config
    
    def evaluate_by_framework(self, test_indices: Set[int], predictions_map: Dict, 
                             ground_truth: Dict, sorted_test: List[int] = None) -> Dict[str, Dict]:
        """Calculate metrics per framework on the test set."""
        # FIX v3.4: Accept pre-computed sorted_test to avoid recomputation
        if sorted_test is None:
            sorted_test = sorted(test_indices)
        test_idx_to_mapped = {test_idx: i for i, test_idx in enumerate(sorted_test)}
        
        framework_regs = defaultdict(list)
        for test_idx in test_indices:
            fw = self.data_gen.get_framework_for_regulation(test_idx)
            mapped_idx = test_idx_to_mapped[test_idx]
            framework_regs[fw].append(mapped_idx)
        
        framework_metrics = {}
        for framework, reg_ids in framework_regs.items():
            fw_ground_truth = {rid: ground_truth[rid] for rid in reg_ids if rid in ground_truth}
            fw_predictions = {rid: predictions_map[rid] for rid in reg_ids if rid in predictions_map}
            
            if not fw_ground_truth:
                continue
            
            recall_values = RigorousMetrics.calculate_recall_at_k(fw_ground_truth, fw_predictions, k=5)
            precision_values = RigorousMetrics.calculate_precision_at_k(fw_ground_truth, fw_predictions, k=5)
            ndcg_values = RigorousMetrics.calculate_ndcg_at_k(fw_ground_truth, fw_predictions, k=5)
            
            recall_mean = np.mean(recall_values) if recall_values else 0.0
            precision_mean = np.mean(precision_values) if precision_values else 0.0
            ndcg_mean = np.mean(ndcg_values) if ndcg_values else 0.0
            
            framework_metrics[framework] = {
                'n_regulations': len(reg_ids),
                'recall_at_5': recall_mean,
                'precision_at_5': precision_mean,
                'f1': RigorousMetrics.calculate_f1(recall_mean, precision_mean),
                'ndcg_at_5': ndcg_mean
            }
        
        return framework_metrics
    
    @staticmethod
    def print_framework_results(framework_metrics: Dict[str, Dict]):
        print("\n    Per-Framework Breakdown (on mixed test set):")
        print("    " + "─" * 70)
        print(f"    {'Framework':<12} {'N_Test':>8} {'Recall@5':>10} {'Prec@5':>10} {'F1':>10} {'NDCG@5':>10}")
        print("    " + "─" * 70)
        for framework, metrics in sorted(framework_metrics.items()):
            print(f"    {framework:<12} {metrics['n_regulations']:>8} "
                  f"{metrics['recall_at_5']:>10.3f} {metrics['precision_at_5']:>10.3f} "
                  f"{metrics['f1']:>10.3f} {metrics['ndcg_at_5']:>10.3f}")
        print("    " + "─" * 70)
        print("    Note: Small N per framework limits statistical power.")


# ==========================================
# BASELINE EVALUATOR (FIXED v3.3)
# ==========================================
class BaselineEvaluator:
    def __init__(self, config: RigorousConfig):
        self.config = config
        self.baselines = {}
    
    def setup_baselines(self):
        print("\n  Setting up baselines...")
        if self.config.RUN_BM25_BASELINE and BM25_AVAILABLE:
            try:
                self.baselines['BM25'] = BM25Retriever()
                print("    ✓ BM25 baseline ready")
            except Exception as e:
                print(f"    ✗ BM25 failed: {e}")
        
        if self.config.RUN_DENSE_ONLY_BASELINE:
            try:
                self.baselines['Generic-Dense'] = DenseOnlyRetriever(self.config.GENERIC_MODEL)
                print("    ✓ Generic dense retriever ready")
            except Exception as e:
                print(f"    ✗ Generic dense failed: {e}")
        
        if self.config.RUN_SINGLE_DOMAIN_BASELINE:
            try:
                self.baselines['LegalBERT-Only'] = SingleDomainRetriever(self.config.LEGALBERT_MODEL, "LegalBERT")
                print("    ✓ LegalBERT-only baseline ready")
            except Exception as e:
                print(f"    ✗ LegalBERT-only failed: {e}")
    
    def evaluate_baseline(self, baseline: BaseRetriever, test_regs: List[str], ground_truth: Dict, k: int = 5) -> Dict:
        """FIX v3.3: Single retrieval call per regulation, store results."""
        predictions_map = {}
        for reg_id, reg_text in enumerate(test_regs):
            results = baseline.retrieve(reg_text, top_k=k)  # Single call
            predictions_map[reg_id] = [r['control_id'] for r in results]
        
        recall_values = RigorousMetrics.calculate_recall_at_k(ground_truth, predictions_map, k=k)
        precision_values = RigorousMetrics.calculate_precision_at_k(ground_truth, predictions_map, k=k)
        ndcg_values = RigorousMetrics.calculate_ndcg_at_k(ground_truth, predictions_map, k=k)
        
        recall_mean = np.mean(recall_values) if recall_values else 0.0
        precision_mean = np.mean(precision_values) if precision_values else 0.0
        ndcg_mean = np.mean(ndcg_values) if ndcg_values else 0.0
        
        return {'recall_at_k': recall_mean, 'precision_at_k': precision_mean, 
                'f1': RigorousMetrics.calculate_f1(recall_mean, precision_mean), 'ndcg_at_k': ndcg_mean}
    
    def run_all_baselines(self, controls: List[str], test_regs: List[str], ground_truth: Dict, k: int = 5) -> Dict[str, Dict]:
        results = {}
        for name, baseline in self.baselines.items():
            print(f"    Evaluating {name}...")
            baseline.ingest_controls(controls)
            results[name] = self.evaluate_baseline(baseline, test_regs, ground_truth, k)
        return results
    
    @staticmethod
    def print_baseline_comparison(baseline_results: Dict[str, Dict], main_results: Dict, k: int = 5):
        print(f"\n  Baseline Comparison (K={k}):")
        print("  " + "═" * 65)
        print(f"  {'System':<25} {'Recall@K':>10} {'Prec@K':>10} {'F1':>10} {'NDCG@K':>10}")
        print("  " + "─" * 65)
        r_mean, r_ci = main_results['recall_at_5']
        p_mean, p_ci = main_results['precision_at_5']
        f_mean, f_ci = main_results['f1']
        n_mean, n_ci = main_results['ndcg_at_5']
        print(f"  {'RAA (Proposed)':<25} {r_mean:>10.3f} {p_mean:>10.3f} {f_mean:>10.3f} {n_mean:>10.3f}")
        print("  " + "─" * 65)
        for name, metrics in sorted(baseline_results.items()):
            print(f"  {name:<25} {metrics['recall_at_k']:>10.3f} {metrics['precision_at_k']:>10.3f} "
                  f"{metrics['f1']:>10.3f} {metrics['ndcg_at_k']:>10.3f}")
        print("  " + "═" * 65)


# ==========================================
# EVALUATION RUNNER
# ==========================================
def run_single_evaluation(config, data_gen, seed, run_baselines=False):
    """FIXED v3.4: Updated for calibration holdout and cached sorted_test."""
    set_all_seeds(seed, strict_determinism=config.STRICT_DETERMINISM)
    
    base_encoder = None
    if config.USE_HARD_NEGATIVES and config.MINE_HARD_NEGATIVES:
        print("    Loading base encoder for hard negative mining...")
        base_transformer = models.Transformer(config.LEGALBERT_MODEL, max_seq_length=256)
        base_pooling = models.Pooling(base_transformer.get_word_embedding_dimension(), pooling_mode='mean')
        base_encoder = SentenceTransformer(modules=[base_transformer, base_pooling])
    
    # FIXED v3.4: Now returns (train_data, cal_reg_indices, actual_train_indices, test_indices)
    train_data, cal_reg_indices, actual_train_indices, test_indices = data_gen.generate_training_data_with_holdout(
        holdout_ratio=config.HOLDOUT_RATIO, seed=seed, use_hard_negatives=config.USE_HARD_NEGATIVES,
        hard_negatives_per_reg=config.HARD_NEGATIVES_PER_REG, mine_hard_negatives=config.MINE_HARD_NEGATIVES,
        encoder_for_mining=base_encoder, use_stratified=config.USE_STRATIFIED_HOLDOUT,
        min_per_framework=config.MIN_TEST_PER_FRAMEWORK,
        calibration_reg_ratio=config.CALIBRATION_REG_RATIO)
    
    if config.USE_STRATIFIED_HOLDOUT:
        fw_counts = defaultdict(int)
        for idx in test_indices:
            fw_counts[data_gen.get_framework_for_regulation(idx)] += 1
        print(f"    Stratified test: {dict(fw_counts)}")
    
    print(f"    Split: {len(actual_train_indices)} train, {len(cal_reg_indices)} cal, {len(test_indices)} test")
    
    exclude_negatives = not config.EVAL_OPEN_WORLD
    # FIXED v3.4: get_validation_set now returns sorted_test
    test_regs, ctrls, ground_truth, eval_to_reg, ctrl_to_eval_idx, eval_idx_to_orig, sorted_test = \
        data_gen.get_validation_set(test_indices, exclude_negatives=exclude_negatives)
    
    print(f"    Data: {len(train_data)} examples, {len(test_regs)} test regs, {len(ctrls)} controls")
    
    tuner = DualModelFineTuner(use_ensemble=config.USE_ENSEMBLE, config=config)
    legal_model, secure_model = tuner.train(train_data)
    
    rag = RigorousRAG(legal_model, secure_model, config=config)
    rag.ingest_controls(ctrls)
    
    # FIXED v3.4: Calibrate on held-out calibration regs (not used in training)
    rag.calibrate_on_retrieval_distribution(cal_reg_indices, data_gen, eval_idx_to_orig)
    
    agent = RigorousAgent(rag, config=config)
    
    # FIXED v3.4: Use cached sorted_test
    predictions_map, y_true_regs, y_pred_regs = {}, [], []
    threshold = config.get_confidence_threshold()
    
    for reg_id, reg_text in enumerate(test_regs):
        result = agent.run(reg_text, reg_id)
        ranked_results = result.get('ranked_results', [])
        predictions_map[reg_id] = [r["control_id"] for r in ranked_results[:config.TOP_K_RERANK]]
        
        y_true_regs.append(sorted_test[reg_id])  # Use cached sorted_test
        if not ranked_results or ranked_results[0]["final_score"] < threshold:
            y_pred_regs.append(ABSTAIN)
        else:
            best_id = ranked_results[0]["control_id"]
            y_pred_regs.append(eval_to_reg.get(best_id, NO_MATCH))
    
    # Calculate metrics
    recall_values = RigorousMetrics.calculate_recall_at_k(ground_truth, predictions_map, k=5)
    precision_values = RigorousMetrics.calculate_precision_at_k(ground_truth, predictions_map, k=5)
    ndcg_values = RigorousMetrics.calculate_ndcg_at_k(ground_truth, predictions_map, k=5)
    
    recall_mean = np.mean(recall_values) if recall_values else 0.0
    precision_mean = np.mean(precision_values) if precision_values else 0.0
    ndcg_mean = np.mean(ndcg_values) if ndcg_values else 0.0
    f1 = RigorousMetrics.calculate_f1(recall_mean, precision_mean)
    
    # FIXED v3.4: Use new kappa and coverage methods
    kappa = RigorousMetrics.calculate_kappa_on_regulations(y_true_regs, y_pred_regs)
    coverage, selective_acc, n_abstain, n_no_match = RigorousMetrics.calculate_coverage_and_accuracy(y_true_regs, y_pred_regs)
    
    # Per-framework evaluation with cached sorted_test
    pf_evaluator = PerFrameworkEvaluator(data_gen, config)
    framework_metrics = pf_evaluator.evaluate_by_framework(test_indices, predictions_map, ground_truth, sorted_test)
    
    # Baselines
    baseline_results = None
    if run_baselines:
        baseline_evaluator = BaselineEvaluator(config)
        baseline_evaluator.setup_baselines()
        baseline_results = baseline_evaluator.run_all_baselines(ctrls, test_regs, ground_truth, k=5)
    
    return {
        'recall_at_5': recall_mean, 'precision_at_5': precision_mean, 'f1': f1, 'ndcg_at_5': ndcg_mean,
        'kappa': kappa, 'coverage': coverage, 'selective_accuracy': selective_acc,
        'n_abstentions': n_abstain, 'n_no_match': n_no_match, 'n_predictions': len(y_pred_regs),
        'framework_metrics': framework_metrics, 'baseline_results': baseline_results,
        'test_indices': test_indices, 'sorted_test': sorted_test
    }


def aggregate_results(all_run_metrics: List[Dict], alpha: float = 0.05) -> Dict:
    """Aggregate metrics with proper CI calculation."""
    keys = ['recall_at_5', 'precision_at_5', 'f1', 'ndcg_at_5', 'kappa', 'coverage', 'selective_accuracy']
    results = {}
    for k in keys:
        values = [m[k] for m in all_run_metrics]
        mean, std, ci = mean_std_ci(values, alpha=alpha)
        results[k] = (mean, ci)
    return results


def aggregate_baseline_results(all_baseline_results: List[Dict], alpha: float = 0.05) -> Dict[str, Dict]:
    """FIX v3.4: Aggregate baseline results across runs with CI."""
    if not all_baseline_results or all_baseline_results[0] is None:
        return {}
    
    baseline_names = list(all_baseline_results[0].keys())
    aggregated = {}
    
    for name in baseline_names:
        metrics = {}
        for metric_key in ['recall_at_k', 'precision_at_k', 'f1', 'ndcg_at_k']:
            values = [run[name][metric_key] for run in all_baseline_results if run and name in run]
            if values:
                mean, std, ci = mean_std_ci(values, alpha=alpha)
                metrics[metric_key] = (mean, ci)
            else:
                metrics[metric_key] = (0.0, 0.0)
        aggregated[name] = metrics
    
    return aggregated


def print_results(results: Dict, n_runs: int, alpha: float = 0.05, world_type: str = ""):
    """FIX v3.4: Dynamic CI percentage label."""
    ci_pct = int((1 - alpha) * 100)
    label = f" ({world_type})" if world_type else ""
    print(f"\n  Results{label} ({n_runs} runs, {ci_pct}% CI):")
    print(f"  ┌────────────────────────────────────────────────────────┐")
    print(f"  │ Recall@5:       {results['recall_at_5'][0]:.3f} ± {results['recall_at_5'][1]:.3f} ({ci_pct}% CI)   │")
    print(f"  │ Precision@5:    {results['precision_at_5'][0]:.3f} ± {results['precision_at_5'][1]:.3f} ({ci_pct}% CI)   │")
    print(f"  │ F1-Score:       {results['f1'][0]:.3f} ± {results['f1'][1]:.3f} ({ci_pct}% CI)   │")
    print(f"  │ NDCG@5:         {results['ndcg_at_5'][0]:.3f} ± {results['ndcg_at_5'][1]:.3f} ({ci_pct}% CI)   │")
    print(f"  │ Kappa:          {results['kappa'][0]:.3f} ± {results['kappa'][1]:.3f} ({ci_pct}% CI)   │")
    print(f"  │ Coverage:       {results['coverage'][0]:.3f} ± {results['coverage'][1]:.3f} ({ci_pct}% CI)   │")
    print(f"  │ Selective Acc:  {results['selective_accuracy'][0]:.3f} ± {results['selective_accuracy'][1]:.3f} ({ci_pct}% CI)   │")
    print(f"  └────────────────────────────────────────────────────────┘")


def print_baseline_comparison(baseline_results: Dict[str, Dict], main_results: Dict, 
                             n_runs: int, alpha: float = 0.05, single_seed: bool = False):
    """FIX v3.4: Print baselines with CI if run across all seeds."""
    ci_pct = int((1 - alpha) * 100)
    seed_note = " (single-seed)" if single_seed else f" ({n_runs} runs, {ci_pct}% CI)"
    
    print(f"\n  Baseline Comparison{seed_note}:")
    print("  " + "═" * 70)
    print(f"  {'System':<25} {'Recall@5':>12} {'Prec@5':>12} {'F1':>12} {'NDCG@5':>12}")
    print("  " + "─" * 70)
    
    r_mean, r_ci = main_results['recall_at_5']
    p_mean, p_ci = main_results['precision_at_5']
    f_mean, f_ci = main_results['f1']
    n_mean, n_ci = main_results['ndcg_at_5']
    print(f"  {'RAA (Proposed)':<25} {r_mean:.3f}±{r_ci:.3f}  {p_mean:.3f}±{p_ci:.3f}  {f_mean:.3f}±{f_ci:.3f}  {n_mean:.3f}±{n_ci:.3f}")
    print("  " + "─" * 70)
    
    for name, metrics in sorted(baseline_results.items()):
        if single_seed:
            print(f"  {name:<25} {metrics['recall_at_k'][0]:.3f}        "
                  f"{metrics['precision_at_k'][0]:.3f}        "
                  f"{metrics['f1'][0]:.3f}        "
                  f"{metrics['ndcg_at_k'][0]:.3f}")
        else:
            r_m, r_c = metrics['recall_at_k']
            p_m, p_c = metrics['precision_at_k']
            f_m, f_c = metrics['f1']
            n_m, n_c = metrics['ndcg_at_k']
            print(f"  {name:<25} {r_m:.3f}±{r_c:.3f}  {p_m:.3f}±{p_c:.3f}  {f_m:.3f}±{f_c:.3f}  {n_m:.3f}±{n_c:.3f}")
    
    print("  " + "═" * 70)


# ==========================================
# COMMAND LINE INTERFACE
# ==========================================
def parse_arguments():
    parser = argparse.ArgumentParser(description='Regulatory Alignment Agent v3.3')
    
    parser.add_argument('--custom-data', type=str, default=None, help='Path to custom data file')
    parser.add_argument('--generate-sample', action='store_true', help='Generate sample data template')
    parser.add_argument('--baselines', action='store_true', help='Run baselines (first run only)')
    parser.add_argument('--baselines-all-runs', action='store_true', help='Run baselines every run for CI')
    parser.add_argument('--runs', type=int, default=5, help='Number of evaluation runs')
    parser.add_argument('--mode', type=str, default='balanced', choices=['conservative', 'balanced', 'aggressive'])
    parser.add_argument('--seed', type=int, default=42, help='Base random seed')
    parser.add_argument('--no-stratified', action='store_true', help='Disable stratified holdout')
    parser.add_argument('--min-per-framework', type=int, default=1, help='Minimum test samples per framework')
    parser.add_argument('--ci-alpha', type=float, default=0.05, help='CI alpha (default 0.05 for 95% CI)')
    parser.add_argument('--calibration-ratio', type=float, default=0.15, help='Fraction of train regs for calibration')
    
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"  Note: Ignoring unknown arguments: {unknown}")
    
    return args


# ==========================================
# MAIN
# ==========================================
if __name__ == "__main__":
    args = parse_arguments()
    
    if args.generate_sample:
        print("\n  Generating sample data template...")
        CustomDataLoader.generate_sample_csv()
        sys.exit(0)
    
    print("\n" + "="*70)
    print(" RIGOROUS REGULATORY ALIGNMENT AGENT - v3.4 (Publication-Ready)")
    print("="*70)
    print("\nv3.4 Fixes:")
    print("  35. ✅ Dynamic CI label (uses actual alpha)")
    print("  36. ✅ Calibration leakage fix (cal regs excluded from training)")
    print("  37. ✅ Apples-to-apples baselines (--baselines-all-runs)")
    print("  38. ✅ n_test clamping (ensures non-empty train)")
    print("  39. ✅ Cached sorted(test_indices)")
    print("  40. ✅ Kappa on regulation IDs only")
    
    config = RigorousConfig()
    config.NUM_EVAL_RUNS = args.runs
    config.USE_STRATIFIED_HOLDOUT = not args.no_stratified
    config.MIN_TEST_PER_FRAMEWORK = args.min_per_framework
    config.CI_ALPHA = args.ci_alpha
    config.CALIBRATION_REG_RATIO = args.calibration_ratio
    config.BASELINES_ALL_RUNS = args.baselines_all_runs
    config.MINE_HARD_NEGATIVES = True
    
    mode_map = {'conservative': OperatingMode.CONSERVATIVE, 'balanced': OperatingMode.BALANCED, 
                'aggressive': OperatingMode.AGGRESSIVE}
    config.mode = mode_map[args.mode]
    
    if args.custom_data:
        print(f"\n  Loading custom data from: {args.custom_data}")
        regulations, controls, mappings = CustomDataLoader.load(args.custom_data)
        stats = CustomDataLoader.validate_data(regulations, controls, mappings)
        print(f"    Loaded: {stats['n_regulations']} regulations, {stats['n_controls']} controls")
        data_gen = CustomDataGenerator(regulations, controls, mappings)
        data_source = "CUSTOM"
    else:
        data_gen = ExpandedDataGenerator()
        data_source = "SYNTHETIC"
        print(f"\n  Using SYNTHETIC data")
    
    print(f"\n  Dataset: {len(data_gen.all_regulations)} regs, {len(data_gen.all_controls_with_families)} controls")
    print(f"  Stratified: {'Yes' if config.USE_STRATIFIED_HOLDOUT else 'No'}")
    print(f"  Calibration holdout: {int(config.CALIBRATION_REG_RATIO*100)}% of train regs")
    print(f"  CI: {int((1-config.CI_ALPHA)*100)}%")
    
    print(f"\n{'='*70}")
    print(f" MODE: {config.mode.value.upper()}")
    print(f"{'='*70}")
    
    all_metrics = []
    all_baseline_results = []
    
    for run_idx in range(config.NUM_EVAL_RUNS):
        print(f"\n  Run {run_idx + 1}/{config.NUM_EVAL_RUNS}:")
        seed = args.seed + run_idx
        
        # FIX v3.4: Run baselines every run if --baselines-all-runs
        run_baselines_this_run = (args.baselines and run_idx == 0) or args.baselines_all_runs
        
        metrics = run_single_evaluation(config, data_gen, seed, run_baselines=run_baselines_this_run)
        all_metrics.append(metrics)
        
        if run_baselines_this_run and metrics.get('baseline_results'):
            all_baseline_results.append(metrics['baseline_results'])
        
        print(f"    Results: Abstain={metrics['n_abstentions']}, Coverage={metrics['coverage']:.2f}, F1={metrics['f1']:.3f}")
    
    # Aggregate results
    results = aggregate_results(all_metrics, alpha=config.CI_ALPHA)
    print_results(results, config.NUM_EVAL_RUNS, alpha=config.CI_ALPHA, world_type="closed-world")
    
    # Per-framework breakdown
    if all_metrics[-1].get('framework_metrics'):
        PerFrameworkEvaluator.print_framework_results(all_metrics[-1]['framework_metrics'])
    
    # Baseline comparison
    if all_baseline_results:
        if args.baselines_all_runs and len(all_baseline_results) > 1:
            # Aggregate baselines across runs with CI
            aggregated_baselines = aggregate_baseline_results(all_baseline_results, alpha=config.CI_ALPHA)
            print_baseline_comparison(aggregated_baselines, results, len(all_baseline_results), 
                                     alpha=config.CI_ALPHA, single_seed=False)
        else:
            # Single-seed baselines
            single_baseline = {name: {k: (v, 0.0) for k, v in metrics.items()} 
                              for name, metrics in all_baseline_results[0].items()}
            print_baseline_comparison(single_baseline, results, 1, 
                                     alpha=config.CI_ALPHA, single_seed=True)
    
    # Summary
    print("\n" + "="*70)
    print(" v3.4 SUMMARY")
    print("="*70)
    print(f"\n  Data Source: {data_source}")
    print(f"  Runs: {config.NUM_EVAL_RUNS}")
    print(f"  CI: {int((1-config.CI_ALPHA)*100)}% (using {'t-distribution' if SCIPY_AVAILABLE else 'normal approx'})")
    print(f"  Calibration: {int(config.CALIBRATION_REG_RATIO*100)}% of train regs held out (no leakage)")
    
    if args.baselines_all_runs:
        print(f"  Baselines: CI computed across {len(all_baseline_results)} runs")
    elif args.baselines:
        print(f"  Baselines: Single-seed (run 1 only)")
    
    if data_source == "SYNTHETIC":
        print("\n  NOTE: Results use SYNTHETIC data. Validate on real data for publication.")

  Note: Ignoring unknown arguments: ['-f', 'C:\\Users\\suman\\AppData\\Roaming\\jupyter\\runtime\\kernel-782d7085-5421-4500-a65f-4633188b1567.json']

 RIGOROUS REGULATORY ALIGNMENT AGENT - v3.4 (Publication-Ready)

v3.4 Fixes:
  35. ✅ Dynamic CI label (uses actual alpha)
  36. ✅ Calibration leakage fix (cal regs excluded from training)
  37. ✅ Apples-to-apples baselines (--baselines-all-runs)
  38. ✅ n_test clamping (ensures non-empty train)
  39. ✅ Cached sorted(test_indices)
  40. ✅ Kappa on regulation IDs only

  Using SYNTHETIC data

  Dataset: 50 regs, 92 controls
  Stratified: Yes
  Calibration holdout: 15% of train regs
  CI: 95%

 MODE: BALANCED

  Run 1/5:
    Loading base encoder for hard negative mining...
    Mining hard negatives via bi-encoder...
    Stratified test: {'GDPR': 1, 'NIST': 2, 'ISO': 1, 'SOX': 1, 'SOC2': 1, 'HIPAA': 1, 'PCI': 2}
    Split: 35 train, 6 cal, 9 test
    Data: 8800 examples, 9 test regs, 84 controls


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\suman\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.0881, 'grad_norm': 0.5033714175224304, 'learning_rate': 4.536046511627907e-06, 'epoch': 0.9090909090909091}
{'loss': 0.0098, 'grad_norm': 0.40316295623779297, 'learning_rate': 3.954651162790698e-06, 'epoch': 1.8181818181818183}
{'loss': 0.0033, 'grad_norm': 0.1788846254348755, 'learning_rate': 3.373255813953489e-06, 'epoch': 2.7272727272727275}
{'loss': 0.0018, 'grad_norm': 0.31822583079338074, 'learning_rate': 2.7918604651162794e-06, 'epoch': 3.6363636363636362}
{'loss': 0.0012, 'grad_norm': 0.09955578297376633, 'learning_rate': 2.2104651162790698e-06, 'epoch': 4.545454545454545}
{'loss': 0.0009, 'grad_norm': 0.09533506631851196, 'learning_rate': 1.6290697674418608e-06, 'epoch': 5.454545454545454}
{'loss': 0.0008, 'grad_norm': 0.15879376232624054, 'learning_rate': 1.0476744186046512e-06, 'epoch': 6.363636363636363}
{'loss': 0.0007, 'grad_norm': 0.052089788019657135, 'learning_rate': 4.6627906976744195e-07, 'epoch': 7.2727272727272725}
{'train_runtime': 19251.7238, 'train_sa

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

...

The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.
The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.
The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.
The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.
The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.
The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.
The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.
The CrossEncoder.predict `activation_fct` argument was renamed and is now deprecated, please use `activation_fn` instead.
The CrossEncoder.predict

    Results: Abstain=2, Coverage=0.78, F1=0.227

  Run 4/5:
    Loading base encoder for hard negative mining...
    Mining hard negatives via bi-encoder...
    Stratified test: {'GDPR': 1, 'NIST': 2, 'ISO': 1, 'SOX': 1, 'SOC2': 1, 'HIPAA': 1, 'PCI': 2}
    Split: 35 train, 6 cal, 9 test
    Data: 8860 examples, 9 test regs, 84 controls


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\suman\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.0822, 'grad_norm': 0.4447064995765686, 'learning_rate': 4.5386216466234976e-06, 'epoch': 0.9025270758122743}
